# verl.workers

- verl.workers 模块是实现其核心计算任务的执行单元。
- 你可以将 Worker 理解为在分布式环境中扮演特定角色的“演员”，它们负责执行具体的计算，如模型推理、训练更新、价值评估等。
- VeRL 的架构设计将<font color='red'>控制流（做什么）</font>与<font color='red'>计算流（怎么做）</font>分离，
    - 而 Worker 正是计算流的具体承载者。


verl.workers 模块是 VeRL 框架的核心执行单元，它通过一套高度模块化和抽象的设计，<font color='red'>实现了对不同训练和推理后端（如 FSDP、Megatron-LM、vLLM、SGLang）的统一支持。</font>

<font color='red'>为了统一多样的后端，verl.workers 引入了一个关键的抽象层——引擎（Engine）</font>


1. BaseEngine 抽象接口

- 这是一个抽象基类，定义了<font color='red'>模型训练</font>和<font color='red'>推理</font>的通用操作契约，
    - 如 initialize()、train_mode()、forward_backward_batch()、optimizer_step() 等。
- 上层的工作者（Worker）只与这个统一的接口交互，而无需关心底层是 FSDP 还是 Megatron。


In [ ]:
class BaseEngine:
    """
    Abstract base class defining the interface for model training engines. Interface is subject to
    change before release.

    Engine implementations must subclass BaseEngine and provide concrete behavior for all methods.
    """

    def initialize(self):
        """
        Instantiate or load the model, optimizer, and learning rate scheduler.

        Should prepare all components necessary for training or evaluation.
        """
        raise NotImplementedError

    @property
    @abstractmethod
    def is_param_offload_enabled(self) -> bool:
        """Whether parameter offloading is enabled."""
        raise NotImplementedError

    @property
    @abstractmethod
    def is_optimizer_offload_enabled(self) -> bool:
        """Whether optimizer offloading is enabled."""
        raise NotImplementedError

    def train_mode(self, **kwargs):
        """
        Context manager entry for switching the engine and model into training mode.

        Usage:
            with engine.train_mode():
                # runs in training mode
        """
        raise NotImplementedError

    def eval_mode(self, **kwargs):
        """
        Context manager entry for switching the engine and model into evaluation mode.

        Usage:
            with engine.eval_mode():
                # runs in evaluation mode
        """
        raise NotImplementedError

    def optimizer_zero_grad(self):
        """
        Zero the gradients of the optimizer.
        """
        raise NotImplementedError

    def optimizer_step(self):
        """
        Perform an optimization step using the optimizer.
        """
        raise NotImplementedError

    def lr_scheduler_step(self):
        """
        Advance the learning rate scheduler by one step.

        Returns:
            current_lr (float or list[float]): Updated learning rate(s).
        """
        raise NotImplementedError

    def forward_backward_batch(self, data: TensorDict, loss_function: Callable, forward_only=False) -> Any:
        """
        Perform a forward pass and optionally a backward pass on a batch of data.

        Args:
            data: The input data for the forward pass, typically containing tensors and metadata.
            loss_function: The loss function to optimize. See `verl.workers.roles.utils.losses` for examples.
            forward_only: If True, perform only the forward pass. If False, perform forward and backward pass.

        Returns:
            Any: The output of the forward pass, which can be used for loss computation or other purposes.
        """
        raise NotImplementedError

    def train_batch(self, data: TensorDict, loss_function: Callable) -> Any:
        """
        Perform a training step on a batch of data.

        Args:
            data: The input data for training, typically containing tensors and metadata.
            loss_function: A function that computes the loss and metrics given a batch and predictions.

        Returns:
            dict[str, torch.Tensor]: A dictionary containing the aggregated training metrics for the batch.
        """
        maybe_fix_3d_position_ids(data)

        self.optimizer_zero_grad()
        outputs = self.forward_backward_batch(data, loss_function, forward_only=False)
        grad_norm = self.optimizer_step()
        if self.is_mp_src_rank_with_outputs():
            assert "grad_norm" not in outputs["metrics"]
            outputs["metrics"]["grad_norm"] = grad_norm
        return outputs

    def infer_batch(self, data: TensorDict, loss_function: Optional[Callable] = None) -> Any:
        """
        Perform inference on a batch of data.

        Args:
            data: The input data for inference, typically containing tensors and metadata.

        Returns:
            Any: The output of the inference, which can be used for predictions or other purposes.
        """
        # see comments from train_batch
        maybe_fix_3d_position_ids(data)

        with torch.no_grad():
            outputs = self.forward_backward_batch(data, loss_function, forward_only=True)
        return outputs

    def get_per_tensor_param(self) -> tuple[Generator[tuple[str, torch.Tensor], None, None], Optional[dict]]:
        """
        Get a generator that yields per-tensor parameters and optional peft config.

        Returns:
            Generator[tuple[str, torch.Tensor]]: A generator that yields tuples of parameter names and tensors.
            Optional[dict]: Optional peft config.
        """
        raise NotImplementedError

    def get_data_parallel_size(self):
        raise NotImplementedError

    def get_data_parallel_rank(self):
        raise NotImplementedError

    def get_data_parallel_group(self):
        raise NotImplementedError

    def to(self, device: str, model: bool = True, optimizer: bool = True, grad: bool = True):
        """
        Move model parameters, optimizer states, or both to the specified device.

        Args:
            device: Target device identifier.
            model: If True, move the model.
            optimizer: If True, move the optimizer states.
            grad: If True, move the gradient buffer.
        """
        if grad:
            assert model, "Gradient buffers must be moved to device along with model parameters"

    def save_checkpoint(
        self,
        local_path: str,
        hdfs_path: Optional[str] = None,
        global_step: int = 0,
        max_ckpt_to_keep: Optional[int] = None,
        **kwargs,
    ) -> None:
        """
        Save model, optimizer, and scheduler states to a checkpoint.

        Args:
            local_path: Local filesystem path to save checkpoint.
            hdfs_path: Optional HDFS path to copy checkpoint.
            global_step: Integer training step number for naming.
            max_ckpt_to_keep: Maximum number of recent checkpoints to retain.
            **kwargs: Arbitrary keyword arguments.
        """
        raise NotImplementedError

    def load_checkpoint(
        self, local_path: str, hdfs_path: Optional[str] = None, del_local_after_load: bool = True, **kwargs
    ) -> None:
        """
        Load model, optimizer, and scheduler states from a checkpoint.

        Args:
            local_path: Local filesystem path of the checkpoint.
            hdfs_path: Optional HDFS path where checkpoint is stored.
            del_local_after_load: Whether to delete local copy after loading.
            **kwargs: Arbitrary keyword arguments.
        """
        raise NotImplementedError

    def is_mp_src_rank_with_outputs(self):
        """
        Whether the current rank is the first rank in model parallel group that contains model outputs
        """
        raise NotImplementedError

    def disable_adapter(self) -> ContextManager:
        """
        Disable all adapters temporarily under the context in the model for LoRA
        """
        return nullcontext()




2. 具体引擎实现

BaseEngine 派生出针对不同后端的实现，如 FSDPEngine 和 MegatronEngine。 <font color='green'>这些具体的引擎类负责封装和实现对应分布式框架的复杂细节。<font>

3. 统一的工作者

- 得益于引擎抽象，VeRL <font color='green'>不再需要为每个后端维护一套独立的 Worker 代码（如旧的 fsdp_worker 和 megatron_worker）。</font>
- 现在，<font color='red'>像 ActorRolloutRefWorker 和 TrainingWorker 这样的通用工作者</font>，<b><font color='green'>通过持有不同的 Engine 实例，就能灵活地支持多种后端，极大地简化了代码维护和新后端的接入。</font></b>



## 主要工作者（Worker）类型

    verl.workers 根据强化学习训练流程中的不同职责，定义了多种工作者：
    
- <b>ActorRolloutRefWorker</b> https://github.com/verl-project/verl/blob/main/verl/workers/engine_workers.py: 这是最核心的工作者，用于强化学习任务。 它集成了
    - <font color='red'>策略网络（Actor）的训练、利用推理后端进行文本生成（Rollout）以及参考模型（Reference Model）的功能。</font>
    - <b>TrainingWorker</b> https://github.com/verl-project/verl/blob/main/verl/workers/engine_workers.py: <font color='red'>一个更通用的训练工作者，包含基础的训练能力，可直接用于监督微调（SFT）等任务。</font>
- <b>CriticWorker</b> : 负责价值网络（Critic）的训练，用于估计状态价值，为策略梯度计算提供基线。
- <b>RewardModelWorker</b> : 用于加载和运行奖励模型，为生成的序列提供奖励分数。
    
    
    <font color='red'>class ActorRolloutRefWorker:</font>
- __init__()
- set_loss_fn()
- to()
- init_model()
- compute_ref_log_prob()
- compute_log_prob()
- update_actor()
- load_checkpoint()
- save_checkpoint()
- update_weights()
- execute_checkpoint_engine()

In [ ]:
    from omegaconf import DictConfig, open_dict
    
class ActorRolloutRefWorker(Worker, DistProfilerExtension):    
    
    
    @register(dispatch_mode=Dispatch.ONE_TO_ALL)
    def init_model(self):
        model_config: HFModelConfig = omega_conf_to_dataclass(self.config.model)

        # 1. build reference model
        if "ref" in self.role:
            # TODO: align ref config with actor config
            '''
            verl 使用 omegaconf 来管理配置。默认情况下配置是只读的，
            open_dict 是一个上下文管理器，允许在代码块内部动态修改配置字典。
            
            Reference Model 在训练中主要负责前向推理（计算 Log Prob），它的批量大小（Batch Size）和微批处理（Micro Batch Size）必须与 Actor 的训练步调保持一致。
            因此，代码将 actor 的 ppo_mini_batch_size 等同步给了 ref。
            
            
            '''
            with open_dict(self.config.ref):
                self.config.ref.ppo_mini_batch_size = self.config.actor.ppo_mini_batch_size
                self.config.ref.ppo_micro_batch_size = self.config.ref.pop("log_prob_micro_batch_size", None)
                self.config.ref.ppo_micro_batch_size_per_gpu = self.config.ref.pop(
                    "log_prob_micro_batch_size_per_gpu", None
                )
                self.config.ref.use_dynamic_bsz = self.config.ref.pop("log_prob_use_dynamic_bsz", False)
                self.config.ref.ppo_max_token_len_per_gpu = self.config.ref.pop("log_prob_max_token_len_per_gpu", None)
            
            '''
            omega_conf_to_dataclass：将普通的配置字典转换为强类型的 Python 数据类（Dataclass），方便后续代码的静态类型检查和调用
            '''
            ref_config: ActorConfig = omega_conf_to_dataclass(self.config.ref)

            # The ref model does not need to enable MTP; force it to false.
            '''
            禁用 MTP（Multi-Token Prediction）：MTP 是一种预测多个后续 Token 的高级训练/推理技术。但对于 Reference Model 来说，
            它只需要老老实实地计算当前生成序列的标准 Log Prob 即可，不需要复杂的多 Token 预测。
            因此，代码通过深拷贝（deepcopy）基础模型配置，并强制关闭 MTP 功能，以节省不必要的计算开销。
            
            '''
            ref_config.model_config = deepcopy(model_config)
            ref_config.model_config.mtp = MtpConfig(enable=False)

            # construct TrainingWorkerConfig
            '''
            这里将模型结构、推理引擎、优化器（虽然 Ref 不更新，但底层 Worker 结构需要该字段）以及检查点相关的配置，
            统一打包进一个 TrainingWorkerConfig 容器中。这体现了 verl 的高度模块化——无论是 Actor 还是 Ref，
            底层都由统一的 TrainingWorker 来驱动
            
            '''
            ref_training_config = TrainingWorkerConfig(
                model_type=ref_config.model_config.get("model_type", "language_model"),
                model_config=ref_config.model_config,
                engine_config=ref_config.engine,
                optimizer_config=ref_config.optim,
                checkpoint_config=ref_config.checkpoint,
            )

            # assign engine configs
            '''
            这一步是将具体的推理引擎参数（如是否使用动态批处理 use_dynamic_bsz、每张卡的最大 Token 长度等）注入到引擎配置中。
            这些参数直接决定了底层（如 Megatron 或 vLLM）在执行前向推理时的显存占用和计算效率。
            '''
            ref_training_config.engine_config.use_dynamic_bsz = self.config.ref.use_dynamic_bsz
            ref_training_config.engine_config.infer_max_token_len_per_gpu = self.config.ref.ppo_max_token_len_per_gpu
            ref_training_config.engine_config.infer_micro_batch_size_per_gpu = (
                self.config.ref.ppo_micro_batch_size_per_gpu
            )
            ref_training_config.engine_config.use_remove_padding = model_config.get("use_remove_padding", False)
            '''
            根据config生成对应的model
            '''
            self.ref = TrainingWorker(config=ref_training_config)
            
            '''
            self.ref.reset()：调用 Worker 的重置方法，通常会完成模型权重的加载、优化器状态的初始化（尽管 Ref 不需要更新，但会占用显存）等操作
            '''
            self.ref.reset()
            
            '''
            self.set_dispatch_collect(...)：这是 verl 分布式调度（HybridFlow）的关键一步。
            它向全局的调度器注册当前 Reference Worker 的通信网格（Device Mesh）信息。
            这样，当 Driver 需要计算 KL 散度时，就能准确地找到这些 Ref 进程，并把数据分发给它们
            '''
            self.set_dispatch_collect(mesh_name="ref", **self.ref.get_dispatch_collect())

        # 2. build actor model
        '''
        
        Actor 是整个 RLHF（强化学习人类反馈）训练中的绝对主角。
        它的任务是根据当前的环境（Prompt）生成回答，并根据奖励信号不断更新自己的参数，以追求更高的奖励分数
        
        '''
        if "actor" in self.role:
            '''
            配置转换：将 self.config.actor 字典转换为强类型的 ActorConfig 数据类，
            并将全局的 model_config（包含模型路径、类型等基础信息）注入进去。
            
            '''
            actor_config: ActorConfig = omega_conf_to_dataclass(self.config.actor)
            actor_config.model_config = model_config
            
            '''
            verl 支持在 RL 训练中引入知识蒸馏。如果开启了 distillation_enabled，代码会加载蒸馏相关的配置。
            这通常用于让 Actor 在追求奖励的同时，不要偏离某个“老师模型”太远，或者加速收敛。
            '''
            distillation_config: Optional[DistillationConfig] = (
                omega_conf_to_dataclass(self.distillation_config) if self.distillation_enabled else None
            )
            '''
            这一步与构建 Reference Model 类似，将模型结构、底层推理/训练引擎、优化器（AdamW等）以及检查点（Checkpoint）相关的配置，
            统一打包进一个 TrainingWorkerConfig 容器中。
            Actor 是需要进行梯度更新的，因此这里的 optimizer_config 会被实际用到。
            '''
            actor_training_config = TrainingWorkerConfig(
                model_type=actor_config.model_config.get("model_type", "language_model"),
                model_config=actor_config.model_config,
                engine_config=actor_config.engine,
                optimizer_config=actor_config.optim,
                checkpoint_config=actor_config.checkpoint,
            )
            '''
            断言检查：Actor 在训练阶段（PPO Update）和推理阶段（计算 Log Prob）必须使用相同的动态批处理策略，
            否则会导致数据形状不匹配，因此代码在这里做了强制检查。
            '''
            assert self.config.actor.use_dynamic_bsz == self.config.rollout.log_prob_use_dynamic_bsz

            # assign engine configs
            actor_training_config.engine_config.use_dynamic_bsz = self.config.actor.use_dynamic_bsz
            
            '''
            max_token_len_per_gpu 和 micro_batch_size_per_gpu 则是为后续的梯度更新（PPO Update）阶段准备的
            '''
            actor_training_config.engine_config.infer_max_token_len_per_gpu = (
                self.config.rollout.log_prob_max_token_len_per_gpu
            )
            actor_training_config.engine_config.infer_micro_batch_size_per_gpu = (
                self.config.rollout.log_prob_micro_batch_size_per_gpu
            )
            actor_training_config.engine_config.max_token_len_per_gpu = self.config.actor.ppo_max_token_len_per_gpu
            actor_training_config.engine_config.micro_batch_size_per_gpu = (
                self.config.actor.ppo_micro_batch_size_per_gpu
            )
            '''
            去填充（Remove Padding）：开启 use_remove_padding 可以在计算时去掉 Prompt 中的无效填充符（Padding），大幅提升长文本训练的计算效率
            '''
            actor_training_config.engine_config.use_remove_padding = model_config.get("use_remove_padding", False)

            '''
            严格的参数校验:
            这段代码确保了配置的完整性。
            如果开启了动态批处理，必须指定每张卡的最大 Token 长度；
            如果没开启，则必须指定固定的微批处理大小（Micro Batch Size）。这能有效防止因配置缺失导致的运行时错误。
            '''
            if self.config.actor.use_dynamic_bsz:
                assert self.config.rollout.log_prob_max_token_len_per_gpu is not None
                assert self.config.actor.ppo_max_token_len_per_gpu is not None
            else:
                assert self.config.rollout.log_prob_micro_batch_size_per_gpu is not None
                assert self.config.actor.ppo_micro_batch_size_per_gpu is not None
            '''
            绑定损失函数（Loss Function）:
            - 如果开启了蒸馏，就使用 distillation_ppo_loss，它会在标准的 PPO 损失基础上，额外增加一项蒸馏损失。
            - 否则，就使用标准的 ppo_loss（包含策略梯度损失和熵正则化等）。
            - 使用 partial 是为了提前把配置参数“固化”到损失函数中，方便后续直接调用。
            
            
            from functools import partial
            '''    
            if self.distillation_enabled:
                self.loss_fn = partial(
                    distillation_ppo_loss, config=actor_config, distillation_config=distillation_config
                )
            else:
                self.loss_fn = partial(ppo_loss, config=actor_config)
            '''
            根据config生成对应的actor
            '''
            self.actor = TrainingWorker(config=actor_training_config)
            self.actor.reset() # 加载预训练权重，初始化优化器状态，为训练做好准备。
            self.actor.set_loss_fn(self.loss_fn)
            
            '''
            向全局调度器注册当前 Actor Worker 的通信网格（Device Mesh）信息。
            这样，Driver 就能准确地指挥这些 Actor 进程进行数据采样和梯度更新。
            '''
            self.set_dispatch_collect(mesh_name="actor", **self.actor.get_dispatch_collect())

        # 3. build rollout engine
        '''
        Rollout 阶段负责“探索”——即让当前的 Actor 模型根据给定的 Prompt 去生成具体的回答（Sequences）。
        由于生成过程通常比较耗时，verl 专门构建了一个独立的推理引擎来高效处理这部分工作。
        '''
        if "rollout" in self.role:
            '''
            配置转换：首先将字典格式的 rollout 配置转换为强类型的 RolloutConfig 数据类
            '''
            rollout_config: RolloutConfig = omega_conf_to_dataclass(self.config.rollout)

            # TODO: move rollout_device_mesh into ServerAdapter
            
            '''
            计算推理并行维度：在 RLHF 中，训练（Actor）和推理（Rollout）的并行策略往往不同。例如，训练时为了省显存可能开启流水线并行（PP），但推理时为了低延迟通常只用张量并行（TP）。代码根据配置计算出了推理阶段需要的 TP、PP 和数据并行（DP）维度。
            '''
            # 3.1 build rollout device mesh (sglang need only)
            infer_tp = rollout_config.tensor_model_parallel_size * rollout_config.data_parallel_size
            infer_pp = rollout_config.pipeline_model_parallel_size
            infer_world_size = infer_tp * infer_pp
            dp = self.world_size // infer_world_size
            assert self.world_size % infer_world_size == 0, (
                f"rollout world_size: {self.world_size} is not divisible by infer_world_size: {infer_world_size}"
            )
            '''
            构建 Device Mesh：init_device_mesh 会为推理引擎创建一个独立的分布式通信组（Device Mesh）。
            这相当于在物理 GPU 集群上，专门为推理任务划分出了一套逻辑上的“虚拟网络”，确保推理进程之间能正确通信，而不会干扰到训练进程。
            '''
            rollout_device_mesh = init_device_mesh(
                get_device_name(), mesh_shape=(dp, infer_tp, infer_pp), mesh_dim_names=["dp", "infer_tp", "infer_pp"]
            )

            # 3.2 initialize rollout engine
            '''
             实例化推理引擎（Rollout Engine）:
             工厂模式获取类：verl 支持多种推理后端（如 vLLM、SGLang 等）。get_rollout_class 会根据配置中的 name（如 "vllm"）和 mode（如 "async"），
             动态返回对应的推理引擎类（例如 vllmAsyncRollout）
             
             
             实例化：传入推理配置、模型基础配置以及刚才构建好的 rollout_device_mesh，正式实例化推理引擎 self.rollout。
             这个引擎将接管后续所有生成数据（Generate Sequences）的任务。
             
             from verl.workers.rollout.base import BaseRollout, get_rollout_class
            '''
            rollout_cls: type[BaseRollout] = get_rollout_class(rollout_config.name, rollout_config.mode)
            self.rollout = rollout_cls(
                config=rollout_config, model_config=model_config, device_mesh=rollout_device_mesh
            )
            '''
            LoRA 与权重同步相关的标志位设置:
            
            这部分代码主要用于处理参数高效微调（PEFT/LoRA）场景下的权重同步逻辑：
            - self.base_sync_done：判断基础模型权重是否已经同步完成。如果 load_format 中不包含 "dummy"（虚拟加载），说明需要加载真实权重，该标志位会被设为 True。
            - self.layered_summon：是否开启“分层召唤”机制。在超大规模模型训练中，为了节省显存，推理引擎可能不需要一次性把整个模型的所有层都加载到 GPU 上，而是按需分层加载。
            - self.peft_merge：判断是否开启了 LoRA 权重的合并（Merge）。在推理阶段，通常需要将训练好的 LoRA 增量权重与基础模型权重合并，以获得完整的推理能力。
            '''
            # used for LoRA (base_sync_done is unused in merge-only mode but kept for Phase 2 adapter path)
            self.base_sync_done: bool = "dummy" not in self.config.rollout.load_format
            self.layered_summon = self.config.rollout.get("layered_summon", False)
            self.peft_merge: bool = model_config.lora.get("merge", False)

        '''
        在大规模 RLHF 训练中，Actor 模型的权重需要在训练（Training）和推理（Rollout）阶段之间频繁同步。
        Checkpoint Engine 的核心作用就是提供一个高效、可靠的机制，来管理这些模型权重的保存、加载和跨进程传输。
        
        '''
        # 4. build checkpoint engine
        '''
        只有当当前 Worker 承担 "actor" 角色时，才需要构建检查点引擎，因为 Actor 是唯一需要更新权重并同步给推理引擎的组件
        '''
        if "actor" in self.role:
            checkpoint_engine_config = omega_conf_to_dataclass(self.config.rollout.checkpoint_engine)
            '''
            提取后端（backend）：获取底层使用的检查点后端（例如某种分布式存储协议或特定的权重传输插件）。
            '''
            backend = checkpoint_engine_config.backend
            '''
            计算桶大小（bucket_size）：<< 20 是位运算，等同于乘以 2的20次方（即 1024 * 1024）。
            这行代码将配置中的“兆字节（MB）”转换成了计算机底层的“字节（Bytes）”，用于控制权重传输时的数据块大小。
            '''
            bucket_size = checkpoint_engine_config.update_weights_bucket_megabytes << 20
            '''
            获取引擎参数：根据选定的后端，提取出专属的引擎配置参数（engine_kwargs）
            '''
            engine_kwargs = checkpoint_engine_config.engine_kwargs.get(backend, {})
            # If custom_backend_module is set, import it so plugins can register
            # in CheckpointEngineRegistry before the backend is instantiated.
            '''
             动态导入外部插件"
             
             - 插件化设计：verl 允许用户使用自定义的检查点后端。如果配置中指定了 custom_backend_module，框架会提前导入这个外部库。
             - 自动注册：导入外部库的目的是触发其内部的注册逻辑，将自定义的后端注册到全局的 CheckpointEngineRegistry（检查点引擎注册表）中，确保下一步能顺利找到并实例化它。
            '''
            import_external_libs(checkpoint_engine_config.custom_backend_module or None)
            '''
            实例化检查点引擎:
            
            - 工厂模式创建：通过 CheckpointEngineRegistry.new 工厂方法，根据 backend 名称动态创建具体的引擎实例。
            - 主节点判定（is_master）：torch.distributed.get_rank() == 0 用于判断当前进程是否为分布式训练中的主进程（Rank 0）。通常只有主进程负责协调权重的持久化存储（如写入磁盘或 HDFS），其他从进程只需负责传输权重给主进程。
            - 传入参数：将计算好的 bucket_size 和专属的 engine_kwargs 传入，完成引擎的最终初始化。
            '''
            self.checkpoint_engine = CheckpointEngineRegistry.new(
                backend, is_master=(torch.distributed.get_rank() == 0), bucket_size=bucket_size, **engine_kwargs
            )

        '''
        强制清理显存缓存:
        
        - 背景：在 verl 的共置（Colocated）设计中，训练进程（Actor）和推理进程（如 vLLM）共享同一批物理 GPU。
        - 痛点：PyTorch 在运行过程中会缓存大量已释放但未归还给操作系统的显存。如果不强制清理，后续启动的 vLLM 推理引擎在调用 cudaMemGetInfo 查询可用显存时，会误以为显存不足，从而导致启动失败或频繁触发昂贵的 CPU 显存交换（Offload）。
        - 解决方案：aggressive_empty_cache(force_sync=True) 会强制触发 CUDA 显存回收，并将缓存归还给操作系统，确保共置的推理引擎能看到真实、充足的空闲显存。
        
        from verl.utils.memory_utils import aggressive_empty_cache
        '''
        # Free cached GPU memory so colocated vLLM processes can see it via cudaMemGetInfo
        aggressive_empty_cache(force_sync=True)


In [ ]:
https://github.com/verl-project/verl/blob/main/verl/workers/rollout/base.py
    
from abc import ABC, abstractmethod    

class BaseRollout(ABC):
    """Base class for rollout."""

    def __init__(
        self,
        config: RolloutConfig,
        model_config: HFModelConfig,
        device_mesh: DeviceMesh,
        *args,
        **kwargs,
    ):
        self.config = omega_conf_to_dataclass(config)
        self.model_config: HFModelConfig = omega_conf_to_dataclass(model_config)
        self.device_mesh = device_mesh

##### <font color='red'>这段代码是 verl 框架高扩展性的体现:</font>

In [ ]:
https://github.com/verl-project/verl/blob/main/verl/workers/rollout/base.py#L90
    
    
_ROLLOUT_REGISTRY = {
    ("vllm", "async"): "verl.workers.rollout.vllm_rollout.ServerAdapter",
    ("sglang", "async"): "verl.workers.rollout.sglang_rollout.sglang_rollout.ServerAdapter",
    ("trtllm", "async"): "verl.workers.rollout.trtllm_rollout.trtllm_rollout.ServerAdapter",
}

'''
输入：接收推理引擎的名字（如 "vllm"）和运行模式（默认为 "async"）。
输出：返回一个继承自 BaseRollout 的具体类（注意返回的是类本身，而不是类的实例）
'''

def get_rollout_class(rollout_name: str, mode: str = "async") -> type[BaseRollout]:
    """Get the rollout class by name.

    Args:
        rollout_name: The name of the rollout.
        mode: The mode of the rollout, async: server mode.

    Returns:
        The rollout class.
    """
    
    '''
    在尝试加载前，先检查用户请求的 (名字, 模式) 组合是否在注册表中。如果不在，直接抛出断言错误，防止后续执行无效操作。
    '''
    assert (rollout_name, mode) in _ROLLOUT_REGISTRY, f"Rollout {rollout_name} with mode {mode} not found"
    
    '''
    提取全限定名（FQDN）:
    从字典中取出对应的字符串路径。例如，如果你传入 ("vllm", "async")，这里取出的就是 "verl.workers.rollout.vllm_rollout.ServerAdapter"。
    '''
    fqdn = _ROLLOUT_REGISTRY[(rollout_name, mode)]
    
    '''
    使用 rsplit(".", 1) 从字符串右边开始分割一次。
    以 "verl.workers.rollout.vllm_rollout.ServerAdapter" 为例：
    - module_name 变为 "verl.workers.rollout.vllm_rollout"（模块路径）
    - class_name 变为 "ServerAdapter"（具体的类名）
    '''
    module_name, class_name = fqdn.rsplit(".", 1)
    
    '''
    动态导入模块:
    
    利用 Python 标准库 importlib，根据字符串路径动态导入对应的模块。
    - 这等同于在代码顶部写 from verl.workers.rollout.vllm_rollout import ServerAdapter，但它是运行时动态执行的。
    '''
    rollout_module = importlib.import_module(module_name)
    
    '''
    使用 getattr 从刚刚导入的模块中，根据类名字符串提取出真正的类对象，并将其返回给调用者。
    '''
    return getattr(rollout_module, class_name)

######  verl.workers.rollout.vllm_rollout.ServerAdapter
- async def update_weights()

In [ ]:
    @torch.no_grad()
    async def update_weights(
        self, weights: Generator[tuple[str, torch.Tensor], None, None], global_steps: int = None, **kwargs
    ):
        """Update model weights via CUDA IPC (fallback to shared memory if IPC not supported) to inference workers."""
        start_time = time.time()

        '''
        这里通过 _execute_method 异步地（non_block=True）向底层的推理引擎（如 vLLM 的 Worker）发送了一个指令，
        告诉它们：“准备一下，马上要更新权重了，请调用你们的 update_weights_from_ipc 方法”。
        返回的 future 是一个异步任务的句柄，程序先把它存起来，等后面权重真正传输完毕后再去确认任务是否完成。
        '''
        future = await self._execute_method(
            "update_weights_from_ipc",
            non_block=True,
            kwargs={**kwargs, "use_shm": self.use_shm},
        )

        '''
        高效传输权重（IPC / 共享内存）:
        
        分桶发送（Bucketed）：大模型的权重非常大（几十 GB），不能一次性塞进通信管道。
        代码根据配置的 bucket_size_mb（比如 128MB 一桶），把权重切分成一个个小包。
        '''
        bucket_size_mb = self.config.checkpoint_engine.update_weights_bucket_megabytes
        
        '''
        IPC / 共享内存（SHM）：因为训练进程（Actor）和推理进程（Rollout）通常部署在同一台机器的不同 GPU 上，直接走网络传输太慢。
        这里使用了 CUDA IPC（进程间通信） 或者 共享内存（Shared Memory） 技术。
        这相当于在两个进程的显存/内存之间开辟了一条“直通高速路”，权重数据可以直接拷贝过去，完全绕过了 CPU 和系统内核，速度极快
        
        from verl.workers.rollout.vllm_rollout.bucketed_weight_transfer import BucketedWeightSender
        
        https://github.com/verl-project/verl/blob/main/verl/workers/rollout/vllm_rollout/bucketed_weight_transfer.py
        '''
        sender = BucketedWeightSender(
            zmq_handle=self.zmq_handle,
            bucket_size_mb=bucket_size_mb,
            use_shm=self.use_shm,
        )
        '''
        await sender.async_send_weights(weights) 会真正开始把这些切好的权重包，通过高速通道异步地推送到推理引擎那边。
        '''
        await sender.async_send_weights(weights)
        
        '''
        等待传输完成并清理 KV Cache
        '''

        if future is not None:
            await future # 等待第2步中推理端接收权重的任务彻底完成。

        # reset prefix cache after updating weights
        if self.rollout_rank == 0:
            '''
            清理 KV Cache：权重都更新了，推理引擎里之前缓存的旧对话历史（KV Cache）就全部失效了，
            必须清空（clear_kv_cache），否则模型会用旧的缓存去匹配新的权重，导致生成乱码。
            '''
            await self.server_handle.clear_kv_cache.remote()
            if global_steps is not None:
                '''
                同步全局步数：如果传入了 global_steps，还会把当前的训练步数同步给推理引擎，方便做日志记录或调试。
                '''
                await self.server_handle.set_global_steps.remote(global_steps)

        '''
        确保只有主进程在权重更新完成后，打印一次本次同步总共花了多少秒
        '''
        if self.replica_rank == 0 and self.rollout_rank == 0:
            logger.info(f"update_weights done, time cost: {time.time() - start_time:.2f}s")

<font color='red'>这段代码是 verl 框架中负责将训练好的模型权重同步给推理引擎（Rollout Engine）的核心异步方法</font>

- 在 RLHF 的训练流程中，Actor 模型每经过一轮训练，权重就会发生变化。
- <font color='red'>为了让推理引擎（如 vLLM 或 SGLang）能够基于最新的策略去采样生成数据，必须将最新的权重高效地传输过去。这个方法就是专门干这件事的。</font>



###### 这段代码是 verl 框架中用于动态加载 Rollout（推理/采样）引擎的工厂函数和注册表。

- 它采用了经典的工厂模式（Factory Pattern）结合注册表模式（Registry Pattern），通过字符串配置来动态导入并返回对应的推理引擎类。这种设计让框架能够轻松支持多种不同的推理后端（如 vLLM、SGLang、TRT-LLM），同时保持上层调用逻辑的高度统一。

下面为你逐层拆解这段代码的执行逻辑：
📝 1. 注册表字典 (_ROLLOUT_REGISTRY)


###### async def async_send_weights()  <font color='red'>会真正开始把这些切好的权重包，通过高速通道异步地推送到推理引擎那边</font>

- 这段代码是 verl 框架中负责实际搬运模型权重数据的核心异步方法。它的主要任务是将训练好的海量模型参数，通过“分桶打包”和“零拷贝”等技术，高效地传输给推理引擎。
- 总结来说，这个方法通过异步迭代、分桶打包、元数据指引和共享内存零拷贝的组合拳，极其高效且稳健地解决了大规模模型权重在进程间同步的难题。

In [ ]:
 async def async_send_weights(self, weights):
        """
        Send weights to the receiver. Accepts a sync generator or async iterator.

        Args:
            weights: Generator or async iterator yielding (name, tensor) pairs
        """
        from verl.workers.rollout.utils import ensure_async_iterator

        try:
            self._init_socket()
            self._init_buffer()

            # send bucket weights
            offset = 0
            bucket_meta: dict[str, TensorMetadata] = {}
            # dtype = PrecisionType.to_dtype(self.config.dtype)
            
            '''
            ensure_async_iterator：这是一个兼容层。无论传入的 weights 是一个普通的 Python 生成器（同步）还是一个异步迭代器，
            它都能将其包装成统一的异步迭代器，确保后续的 async for 循环能顺畅执行。
            '''
            async for name, weight in ensure_async_iterator(weights):
                '''
                
                为什么保留 FP32 全精度？ 代码中特意注释掉了将权重强制转换精度（如转为 BF16）的操作。
                这是因为大模型中某些特殊参数（例如 MoE 架构中的门控网络 moe gate）必须保持 FP32 全精度才能保证模型的稳定性和效果。
                如果在这里强制降精度，可能会导致模型推理出错。
                因此，框架选择保留原始精度，将精度转换的任务交给推理端（Rollout）按需处理。
                
                '''
                # model parameters are in fp32 full precision
                # (vermouth1992) we should not force cast weight here because some parameters
                # (such as moe gate) have to keep fp32 precision. If a weight is bf16 in the rollout side,
                # the rollout should automatically cast on demand. However, this would incur a higher weight
                # transfer volume.
                # weight = weight.to(dtype, non_blocking=True)

                # fill the tensor bucket
                '''
                
                核心机制：分桶打包（Bucketing）
                大模型的权重动辄几十上百 GB，不能一次性塞进通信管道。代码采用了“装桶”的策略：
                
                - offset 和 bucket_size：offset 记录当前“桶”（预分配的共享内存缓冲区）已经装了多少字节。每当要放入一个新权重时，先判断桶里剩下的空间够不够。
                
                - 满桶发送：如果桶满了（且桶里已经有数据），就先调用 get_torch_device().synchronize() 确保 GPU 上的数据拷贝操作全部完成。
                然后通过 Socket 发送这个桶的“清单”（bucket_meta，记录了桶里有哪些权重、形状是什么、在缓冲区的哪个位置），
                并等待接收方的确认（self.socket.recv()）。
                发完后，清空清单和偏移量，准备装下一个桶。
                '''
                if offset + weight.nbytes > self.bucket_size and len(bucket_meta) > 0:
                    get_torch_device().synchronize()
                    self.socket.send_pyobj({"bucket_meta": bucket_meta, "is_last": False})
                    self.socket.recv()
                    bucket_meta = {}
                    offset = 0

                '''
                超大权重的特殊处理:
                
                如果遇到单个权重就比整个桶还大的极端情况（例如某些巨型 Embedding 层），普通的装桶逻辑就行不通了。
                此时，代码会断言当前不能是使用共享内存（SHM）的模式，
                并调用 _direct_send_large_weight 走专门的“超大件物流通道”直接发送该权重，发完后跳过后续的装桶逻辑，继续处理下一个权重。
                '''
                if offset + weight.nbytes > self.bucket_size:
                    assert not self.use_shm, (
                        f"Weight {name}({weight.shape}, {weight.dtype}) is too large to fit in the bucket."
                        f"Please increase rollout.update_weights_bucket_megabytes({self.bucket_size_mb} MB)."
                    )
                    self._direct_send_large_weight(name, weight)
                    continue

                '''
                零拷贝写入共享内存:
                
                记录元数据：把当前权重的名字、形状、数据类型以及它在桶里的起始位置（offset）记录在 bucket_meta 字典中。接收方拿到这个字典，就知道怎么去缓冲区里把权重还原出来。
                
                零拷贝搬运：self.buffer 是预先分配好的共享内存（或 CUDA IPC 显存）。
                通过 .copy_() 方法，直接把权重的二进制数据（先展平再转为 uint8 视角）异步地（non_blocking=True）拷贝到共享内存的指定位置。
                因为是直接在共享内存上操作，接收方进程可以直接读取这块内存，完全省去了“发送端打包 -> 网络传输 -> 接收端解包”的冗余拷贝过程，速度极快。
                '''
                bucket_meta[name] = {
                    "name": name,
                    "shape": weight.shape,
                    "dtype": weight.dtype,
                    "offset": offset,
                    "handle": None,
                }
                self.buffer[offset : offset + weight.nbytes].copy_(weight.view(-1).view(torch.uint8), non_blocking=True)
                offset += weight.nbytes

            '''
            发送最后一个桶并清理:
            
            - 循环结束后，桶里可能还剩下一部分没发出去的权重。代码会再次同步设备，
            发送最后一个桶的元数据，并带上 is_last: True 的标志，告诉接收方：“所有权重都发完了，可以开始干活了”。
            
            - finally 块中的 _cleanup() 确保了无论传输过程中是否发生异常，底层的 Socket 连接和共享内存资源都能被正确释放，防止资源泄漏。
            '''
            # send the last bucket
            get_torch_device().synchronize()
            
            self.socket.send_pyobj({"bucket_meta": bucket_meta, "is_last": True})
            self.socket.recv()
        finally:
            self._cleanup()

###### _init_socket()

In [ ]:
import zmq


class BucketedWeightSender:
    
    """
    Send model weights via bucketed IPC transfer over ZMQ.

    Packs weight tensors into a fixed-size communication buffer and sends them
    in buckets to the receiver. Supports CUDA IPC and shared memory fallback.

    Args:
        zmq_handle: ZMQ IPC socket path (e.g., "ipc:///tmp/rl-colocate-zmq-<uuid>.sock")
        bucket_size_mb: Communication buffer size in MB
        use_shm: Use shared memory instead of CUDA IPC (for NPU compatibility)
    """

    def __init__(
        self,
        zmq_handle: str,
        bucket_size_mb: int = 512,
        use_shm: bool = False,
    ):
        self.zmq_handle = zmq_handle
        self.bucket_size_mb = bucket_size_mb
        self.bucket_size = int(bucket_size_mb) << 20
        self.use_shm = use_shm

        '''
        
        '''
        self.zmq_context = zmq.Context.instance()
        self.socket = None
        self.buffer = None
        self.shm = None
        
        
    def _init_socket(self):
        """Initialize ZMQ REQ socket and bind."""
        if self.zmq_handle.startswith("ipc://"):
            ipc_path = self.zmq_handle[len("ipc://") :]
            try:
                os.remove(ipc_path)
            except OSError:
                pass
            
        self.socket = self.zmq_context.socket(zmq.REQ)
        self.socket.bind(self.zmq_handle)

###### @register(dispatch_mode=Dispatch.ONE_TO_ALL) 
- <font color='red'>是 verl 分布式调度的关键：它表示当 Driver（主控进程）调用这个方法时，指令会被广播给该 WorkerGroup 中的每一个分布式进程</font>（例如在 8 张卡上训练，这 8 个进程都会执行一遍 init_model），确保所有节点都完成模型的加载和环境的搭建。

1. <font color='green'>构建 Reference Model（参考模型）</font>
    - 作用：Reference Model 是训练前的原始模型（通常是 SFT 模型），在 RL 训练中保持冻结，用于计算 KL 散度，防止策略模型（Actor）偏离原始分布太远。
2. <font color='green'>构建 Actor Model（策略模型）</font>
    - 作用：Actor 是 RLHF 中真正被训练和更新的核心模型
3. <font color='green'>构建 Rollout Engine（推理/采样引擎）</font>
    - 作用： <font color='red'>Rollout 负责接收 Prompt，通过模型生成回复（采样），这是 RL 训练中非常耗时的“探索”阶段</font>
    - 工厂模式实例化：使用 get_rollout_class 根据配置（如 vLLM 或 SGLang）动态获取推理引擎类，并传入模型配置和设备网格进行实例化，赋值给 self.rollout。
4. <font color='green'>构建 Checkpoint Engine（检查点引擎）</font>
    - 作用：负责在训练过程中高效地保存和加载模型权重，特别是在多节点环境下实现权重的快速同步
    - 主进程判定：通过 torch.distributed.get_rank() == 0 判断当前是否为主节点，通常只有主节点负责协调权重的持久化存储。
    
###### TrainingWorker根据config生成model
- self.actor = TrainingWorker(config=actor_training_config)
- self.ref = TrainingWorker(config=ref_training_config)



### 🧩 核心 Worker 角色
1. <font color='green'>Actor Worker (verl.workers.actor)</font>

    这是 RL 训练中的核心角色，<font color='red'>负责策略（Policy）的执行和更新。</font>
- 功能：
    - <font color='red'>生成 (Rollout)：</font>接收提示词（prompts），利用当前策略模型生成文本序列。
    - <font color='red'>训练 (Update)：</font>根据计算出的优势（Advantage）等信号，更新自身的模型参数。
- 实现：<font color='red'>为了支持不同的并行策略，Actor Worker 有多种实现，如 DataParallelActor 和 MegatronActor</font>

2. <font color='green'>Critic Worker (verl.workers.critic)</font>

    Critic 负责评估 Actor 生成结果的价值。
- 功能：
    - 价值估计：接收 Actor 生成的序列，预测其累积奖励（Value）。
    - 训练 (Update)：<font color='red'>通过比较预测价值和实际回报（Return），更新自身的价值网络（Value Network）参数，以更准确地评估状态。</font>
- 实现：同样，Critic 也有 DataParallelCritic 和 MegatronCritic 等实现。

3. <font color='green'>Reward Model Worker (verl.workers.reward_model)</font>

这个 Worker 专门用于提供奖励信号。
    - 功能：<font color='red'>对 Actor 生成的文本序列进行打分，这个分数是 RL 算法（如 PPO）优化策略的关键依据。</font>
- 实现：通常是一个独立的模型，其实现也支持分布式部署。

4. <font color='green'>Rollout Worker (verl.workers.rollout)</font>

Rollout Worker 专注于高效的文本生成。
- 功能：为了追求极致的推理吞吐，VeRL 允许将生成任务卸载到专门的推理引擎上。Rollout Worker 就是这些引擎的封装。
- 实现：例如，<font color='red'>vllm_rollout 模块就是对高性能推理库 vLLM 的封装，它利用 vLLM 的连续批处理和 PagedAttention 等技术来加速生成过程。</font>

## ⚙️ 管理与调度机制

VeRL 通过一套精巧的机制来管理和调度这些 Worker，使其能够高效协同。

1. <font color='red'>基于 Ray 的分布式执行</font>

VeRL 构建在 Ray 之上，每个 Worker（如 Actor、Critic）在启动时都会被实例化为一个 Ray Actor。
- 资源隔离：通过 Ray 的 PlacementGroup，可以精确地将不同的 Worker 分配到指定的 GPU 上。例如，你可以将计算密集的 Actor 放在高性能 GPU 集群上，而将 Critic 放在另一组 GPU 上。
- 独立运行：每个 Worker 作为独立的 Ray Actor 运行，拥有自己的内存空间和执行线程，通过远程过程调用（RPC）进行通信，避免了单点瓶颈。

2. <font color='red'>分片管理器 (verl.workers.sharding_manager)</font>

这是 VeRL 实现高性能的关键组件之一，<font color='red'>它解决了 RL 训练中模型在不同阶段需要不同并行策略的难题。</font>
- 问题：Actor 在“生成”阶段适合使用张量并行（TP），而在“训练”阶段则更适合数据并行（DP）或 FSDP。传统方法在切换时需要保存和加载整个模型状态，开销巨大。
- 解决方案：sharding_manager 负责管理模型参数在不同并行模式（如从 TP 切换到 FSDP）之间的动态重分片（Re-sharding）。它通过预规划和内存复用技术，极大地减少了参数重新分布的通信开销和显存占用，实现了“零冗余”内存管理。

3. <font color='red'>数据传输协议 (verl.workers 内部)</font>

为了协调不同 Worker 之间的数据流动，VeRL 内部设计了一套数据传输协议。

- 分发 (Dispatch)：将数据（如一个批次的提示词）按照预设的并行策略（如按 DP 维度切分）分发给各个 Worker。
- 收集 (Collect)：将各个 Worker 的计算结果（如生成的序列）收集并聚合起来，传递给下一个阶段的 Worker。









### 推理后端

 为了实现高效的文本生成（Rollout），verl.workers 通过适配器（Adapter）模式集成了高性能推理引擎：
 
 | 推理后端 | 集成方式 | 优势 |
| :--- | :--- | :--- |
| vLLM | `vllm_rollout` / `VllmReplica` | 提供高吞吐量的连续批处理生成能力。 |
| SGLang | `sglang_rollout` / `SglangReplica` | 在结构化输出和长上下文推理场景下性能更优。 |
| Hugging Face | `HFRollout` | 方便用于调试或小模型验证。 |
